# Génération de format de chargement de tarifs

Génération de deux fichiers parquet pour les tarifs et pour les liens tarif-pdc.

In [1]:
from datetime import datetime, timezone
import sys
from pathlib import Path
import json
import pandas as pd
# remonte à la racine du projet OCPI
sys.path.append(str(Path().resolve().parents[0]))
from source.tariff import TariffObject, TariffDimensionTypeEnum, TariffElement, TariffElements, PriceComponent

In [2]:
# dashboard infrastructure - export / question stations-pdc-puissance

statiques = pd.read_csv("../data/qualicharge/donnees_statiques___liste_stations_pdcs_puissance_2026-06-24T10_05.csv", sep=",")
statiques_50 = statiques[statiques["puissance_nominale"] >= 50]
total_50 = len(statiques_50)
total_50

36593

In [3]:
def tariffs_to_dataframe(tariffs: list[dict], schema: dict, verbose: bool = False) -> pd.DataFrame:
    original_id = []
    original_last_updated = []
    raw = []
    start= []
    end = []
    for tarif in tariffs:
        assert TariffObject.is_valid_json(schema, tarif, verbose=verbose), f"Tariff {tarif['id']} is not valid according to the schema"
        original_id.append(tarif["country_code"] + tarif["party_id"] + tarif["id"])
        original_last_updated.append(tarif["last_updated"])
        raw.append(tarif)
        start.append(max(datetime.fromisoformat(tarif["last_updated"]), datetime.fromisoformat(tarif.get("start_date_time", datetime(1970, 1, 1, tzinfo=timezone.utc).isoformat()))))
        end.append(datetime.fromisoformat(tarif["end_date_time"]) if tarif.get("end_date_time") else None)
    return pd.DataFrame({
        "original_id": original_id,
        "original_last_updated": original_last_updated,
        "raw": raw,
        "start": start,
        "end": end})

In [4]:
def tariffs_from_OCPI(locations: list, tariffs: list, schema: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    tariff = tariffs_to_dataframe(tariffs, schema)  
    id_pdc_itinerance = []
    id_tariff = []
    for location in locations:
        for evse in location["evses"]:
            id_pdc_itinerance.append(evse["evse_id"].replace("*", ""))
            tariff_id = location["country_code"] + location["party_id"]
            tariff_id+= evse["connectors"][0]["tariff_ids"][0]
            if tariff_id not in tariff["original_id"].values:
                print(f"Tariff ID {tariff_id} not found in original tariffs")
            id_tariff.append(tariff_id)
    tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": id_pdc_itinerance,
        "id_tariff": id_tariff})
    return (tariff, tariffpdc)

In [5]:
def tariff_simple(country_code: str, party_id: str, id: str, last_updated: str, dimensions: list[str], prices: list[float], start_date_time: str=None, end_date_time: str=None):
    price_components = []
    for i in range(len(dimensions)):
        price_component = PriceComponent(type=TariffDimensionTypeEnum(dimensions[i]), price=prices[i], step_size=1)
        price_components.append(price_component)
    tariff_elements = TariffElements(
        root=[TariffElement(price_components=price_components)]
    )
    tariff = TariffObject(
        country_code=country_code,
        party_id=party_id,
        id=id,
        elements=tariff_elements,
        last_updated=last_updated,
        start_date_time=start_date_time,
        end_date_time=end_date_time
    )
    return json.loads(tariff.model_dump_json(exclude_none=True))

In [6]:
def tariffs_to_text(tariffs: pd.DataFrame, operator: str) -> None:
    text = "# Tarifs au format texte des tarifs " + operator + "\n\n"
    for tarif in tariffs["raw"]:
        tariff = TariffObject.model_validate(tarif)
        text += tariff.to_text() + "\n"
    with open(f"../data/{operator}/tariffs_text.md", "w", encoding="utf-8") as f:
        f.write(text)

In [7]:
def export_tariffs(tariffs: pd.DataFrame, tariffpdc: pd.DataFrame, operator: str) -> None:
    tariffpdc_50 = tariffpdc[tariffpdc["id_pdc_itinerance"].isin(statiques_50["id_pdc_itinerance"])]
    tariffpdc = tariffpdc.dropna()
    print(f"len tariff, tariffpdc, tariffpdc_50 et % : {len(tariffs)}, {len(tariffpdc)}, {len(tariffpdc_50)}, {len(tariffpdc_50)/total_50*100:.2f} %")
    tariffs_to_text(tariffs, operator)
    tariffs.to_parquet(f"../data/{operator}/qualicharge_tariff.parquet", index=False)
    tariffpdc.to_parquet(f"../data/{operator}/qualicharge_tariffpdc.parquet", index=False)
    tariffpdc.to_csv(f"../data/{operator}/qualicharge_tariffpdc.csv", index=False)

## Tesla

In [8]:
operator = "tesla"

# with open(f'../data/{operator}/FR_Supercharging_locations.json', 'r', encoding='utf-8') as f:
with open(f'../data/{operator}/FR_Open_Locations.json', 'r', encoding='utf-8') as f:
    tesla_locations = json.load(f)
#with open(f'../data/{operator}/FR_Supercharging_NTSLA_tariffs.json', 'r', encoding='utf-8') as f:
with open(f'../data/{operator}/FR_NTSLA_tariffs.json', 'r', encoding='utf-8') as f:
    tesla_tariffs = json.load(f)
with open("../source/schema.json") as f:
    schema = json.load(f)
tariffs, tariffpdc = tariffs_from_OCPI(tesla_locations["data"], tesla_tariffs["data"], schema)

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc, tariffpdc_50 et % : 273, 4063, 4063, 11.10 %


In [9]:
# tariffpdc

## Fastned

In [10]:
operator = "fastned"
country_code = "FR"
party_id = "FAS"
id = "tarif_unique_1"

with open("../source/schema.json") as f:
    schema = json.load(f)
tariff = tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=id,
    last_updated="2024-06-01T12:00:00Z",
    dimensions=["ENERGY"],
    prices=[0.5083]) # 0.61€/kWh TTC -> 0.61/1.2 = 0.5083€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_fastned = statiques[statiques["id_pdc_itinerance"].str.startswith("FRFAS", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_fastned["id_pdc_itinerance"].tolist(),
        "id_tariff": [f"{country_code}{party_id}{id}"] * len(statiques_fastned)})

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc, tariffpdc_50 et % : 1, 457, 457, 1.25 %


In [11]:
# print(tariffs["raw"][0], "\n")
# print(tariffs, "\n")
# print(tariffpdc)


## Atlante

In [12]:
operator = "atlante"
country_code = "FR"
party_id = "ATL"

tariffs_pdc = pd.read_csv(f"../data/{operator}/current_tariffs.csv", sep=",") # .rename(columns={"rateid": "id_tariff"})
tariffs_pdc["id_tariff"] = tariffs_pdc["rateid"].apply(lambda x: "FRATL" + x)
tariffpdc = tariffs_pdc[["id_pdc_itinerance", "id_tariff"]]

tariffs = tariffs_pdc.drop_duplicates(subset=["id_tariff"]).reset_index(drop=True)
tariffs["original_id"] = tariffs["id_tariff"]
tariffs["original_last_updated"] = tariffs["rate_date_modified"]
tariffs["raw"] = tariffs.apply(lambda row: tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=row["rateid"],
    last_updated=row["rate_date_modified"] + "Z",
    dimensions=["ENERGY", "TIME", "PARKING_TIME"],
    prices=[row["costkwh"]/1.2, row["costchargingtime"]/1.2, row["costparkingtime"]/1.2]), axis=1)
tariffs["start"] = tariffs["rate_date_modified"]
tariffs["end"] = None
tariffs = tariffs[["original_id", "original_last_updated", "raw", "start", "end"]]

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc, tariffpdc_50 et % : 9, 1392, 1118, 3.06 %


In [13]:
# print(tariffs["raw"][0], "\n")
# print(tariffs, "\n")
# print(tariffpdc)

## Lidl

In [14]:
operator = "lidl"
country_code = "FR"
party_id = "LDL"
id = "TARIFF_LIDL_PLUS_DC"

with open("../source/schema.json") as f:
    schema = json.load(f)
tariff = tariff_simple(
    country_code=country_code,
    party_id=party_id,
    id=id,
    last_updated="2026-04-22T14:41:00Z",
    dimensions=["ENERGY"],
    prices=[0.325]) # 0.39€/kWh TTC -> 0.39/1.2 = 0.325€/kWh HT)
tariffs = tariffs_to_dataframe([tariff], schema, verbose=False)

statiques_lidl = statiques[statiques["id_pdc_itinerance"].str.startswith("FRLDL", na=False)]
tariffpdc = pd.DataFrame({
        "id_pdc_itinerance": statiques_lidl["id_pdc_itinerance"].tolist(),
        "id_tariff": [f"{country_code}{party_id}{id}"] * len(statiques_lidl)})

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc, tariffpdc_50 et % : 1, 5083, 1947, 5.32 %


In [15]:
# print(tariffs["raw"][0], "\n")
# print(tariffs, "\n")
# print(tariffpdc)


## Total

In [16]:
operator = "total"

with open(f"../data/{operator}/tariffs_qualicharge 1.json", 'r', encoding='utf-8') as f:
    total_tarifs = json.load(f)
tariffs = tariffs_to_dataframe(total_tarifs, schema, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]
#print(tariffs_extended, "\n")

tariffs_pdc = pd.read_csv(f"../data/{operator}/lien_tariff.csv", sep=",")
tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="id_tariff", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})
#print(tariff_pdc, "\n")

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc, tariffpdc_50 et % : 75, 4837, 2553, 6.98 %


In [17]:
# print(tariffs["raw"][0], "\n")
# print(tariffs, "\n")
# print(tariffpdc)


## Izivia

In [18]:
operator = "izivia"

izivia_tarifs = []
jsons = Path(f"../data/{operator}/jsons")
for file in jsons.iterdir():
    with open(file, 'r', encoding='utf-8') as f:
        izivia_tarifs.append(json.load(f))
# print(f"len izivia_tarifs : {len(izivia_tarifs)}")

tariffs = tariffs_to_dataframe(izivia_tarifs, schema, verbose=False)
tariffs_extended = tariffs.copy()
tariffs_extended["id"] = tariffs_extended["original_id"].str[5:]
#print(tariffs_extended, "\n")

tariffs_pdc = pd.read_csv(f"../data/{operator}/edf-bs-tariffs-by-evse.csv", sep=";")
tariffs_pdc.rename(columns={"id_point_de_charge": "id_pdc_itinerance", "tariff_id": "id_tariff"}, inplace=True)

tariffs_pdc_extended = pd.merge(tariffs_pdc, tariffs_extended, how="left", left_on="id_tariff", right_on="id")
tariffpdc = tariffs_pdc_extended[["id_pdc_itinerance", "original_id"]].rename(columns={"original_id": "id_tariff"})
# print(tariffpdc, "\n")

export_tariffs(tariffs, tariffpdc, operator)


len tariff, tariffpdc, tariffpdc_50 et % : 16, 2485, 2152, 5.88 %


In [19]:
# print(tariffs["raw"][0], "\n")
# print(tariffs, "\n")
# print(tariffpdc)


## Driveco

In [20]:
operator = "driveco"
country_code = "FR"
party_id = "DRV"

with open(f'../data/{operator}/data-locations-driveco.json', 'r', encoding='utf-8') as f:
    driveco_locations = json.load(f)
with open(f'../data/{operator}/data-tariffs-driveco.json', 'r', encoding='utf-8') as f:
    driveco_tariffs = json.load(f)
with open("../source/schema.json") as f:
    schema = json.load(f)
locations_json = driveco_locations["data"]
for location in locations_json:
    location["country_code"] = country_code if "country_code" not in location else location["country_code"]
    location["party_id"] = party_id if "party_id" not in location else location["party_id"]
    # correction : tariff_id : value -> tariff_ids : [value]
    for evse in location["evses"]:
        for connector in evse["connectors"]:
            if "tariff_ids" not in connector:
                connector["tariff_ids"] = [connector["tariff_id"]]
tariffs_json = driveco_tariffs["data"]
for tarif in tariffs_json:
    tarif["country_code"] = country_code if "country_code" not in tarif else tarif["country_code"]
    tarif["party_id"] = party_id if "party_id" not in tarif else tarif["party_id"]
    # correction : step_size= 0 -> step_size= 1
    for element in tarif["elements"]:
        for price_component in element["price_components"]:
            if price_component["step_size"] == 0:
                price_component["step_size"] = 1
tariffs, tariffpdc = tariffs_from_OCPI(driveco_locations["data"], tariffs_json, schema)

export_tariffs(tariffs, tariffpdc, operator)

len tariff, tariffpdc, tariffpdc_50 et % : 85, 1730, 998, 2.73 %


In [21]:

with open('../data/driveco/Tarifs.json', 'r', encoding='utf-8') as f:
    driveco_tarifs = json.load(f)
len(driveco_tarifs)

85

# a supprimer

In [22]:
import subprocess
import re
import socket

print("Recherche des équipements sur le réseau local (Version robuste)...")

# On récupère le résultat brut sous forme de bytes (pas de problème d'encodage)
process = subprocess.run(["arp", "-a"], capture_output=True)
output_bytes = process.stdout

if not output_bytes:
    print("Erreur : Impossible de récupérer les données du réseau.")
else:
    # On cherche les IP directement dans les octets (le pattern a un 'b' devant)
    ip_matches = re.findall(rb"(?:[0-9]{1,3}\.){3}[0-9]{1,3}", output_bytes)
    
    # On convertit les octets trouvés en chaînes de caractères classiques
    ip_addresses = [ip.decode('ascii') for ip in ip_matches]

    print(f"\n{'Adresse IP':<18} | {'Nom de l\'appareil (Type)':<30}")
    print("-" * 53)

    # Pour chaque IP unique trouvée
    for ip in set(ip_addresses):
        # On ignore les adresses de diffusion (broadcast / multicast)
        if ip.endswith(".255") or ip.startswith("224.") or ip == "255.255.255.255":
            continue
            
        try:
            # Tente de récupérer le nom de l'hôte via le DNS local
            hostname = socket.gethostbyaddr(ip)[0]
        except socket.herror:
            hostname = "Inconnu (Nom non diffusé)"
            
        print(f"{ip:<18} | {hostname:<30}")

Recherche des équipements sur le réseau local (Version robuste)...

Adresse IP         | Nom de l'appareil (Type)      
-----------------------------------------------------
192.168.31.1       | XiaoQiang                     
172.23.227.97      | Inconnu (Nom non diffusé)     
239.255.255.250    | Inconnu (Nom non diffusé)     
172.29.0.1         | LOCO-LABS.mshome.net          
172.23.224.1       | LOCO-LABS                     
192.168.31.19      | host.docker.internal          
